# 🇹🇷 AUTSL 10 Sınıf: Colab Model Eğitimi ve Karşılaştırması
Bu notebook, **AUTSL 10 Sınıf** veri kümesi için Drive'a yüklediğiniz hazır önbellek paketi (`autsl_10_caches.zip`) üzerinden **20 Epochluk Kontrollü Model Eğitimini** ve resmi **Val Değerlendirmesini** yürütür.

> **Karşılaştırılacak İki Varyant:**
> 1. **Temel Kontrol Kolu (`autsl_10_base16_screen`):** Standart 16 kare MediaPipe önbelleği (`autsl_10_features`).
> 2. **Aday Kolu (`autsl_10_centerrepair_screen`):** 1,5x genişletilmiş kutu + kısa boşluk merkez onarımlı önbellek (`autsl_10_features_centerrepair_strict`).
>
> Her iki model de aynı T4 GPU, aynı `seed=42`, aynı mimari (MobileNetV3-Small) ve aynı 20 epoch bütçesiyle eğitilip resmi val setinde karşılaştırılır.

### 1. Google Drive Bağlantısı ve Kütüphanelerin Kurulumu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gerekli kütüphaneleri yükle
!pip install -q pyyaml rich opencv-python matplotlib scikit-learn tqdm mediapipe

import os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("✅ Google Drive bağlandı, bağımlılıklar hazır.")

### 2. Proje Kodlarının Colab Yerel Hızlı Diskine Kopyalanması

In [ ]:
import os, shutil

drive_project = "/content/drive/MyDrive/BitirmeProjesi"
local_project = "/content/BitirmeProjesi"

os.makedirs(local_project, exist_ok=True)

# Kod, config ve veri klasörlerini yerel diske kopyala
for folder in ["src", "scripts", "configs", "data"]:
    src_f = os.path.join(drive_project, folder)
    dst_f = os.path.join(local_project, folder)
    if os.path.exists(src_f):
        if os.path.exists(dst_f):
            shutil.rmtree(dst_f)
        shutil.copytree(src_f, dst_f)
        print(f"✅ {folder} klasörü yerel diske kopyalandı.")
    else:
        print(f"⚠️ Uyarı: {src_f} klasörü Drive'da bulunamadı!")

# Model ağırlıkları ve logların kaybolmaması için outputs klasörünü Drive'a bağla
os.makedirs(f"{drive_project}/outputs", exist_ok=True)
if not os.path.exists(f"{local_project}/outputs"):
    os.symlink(f"{drive_project}/outputs", f"{local_project}/outputs")

print("✅ Kodlar ve Drive bağlantısı hazır!")

### 3. Önbellek Zip Paketini Açma (`autsl_10_caches.zip`)
*Drive'daki tek parça `autsl_10_caches.zip` dosyasını Colab'ın ultra hızlı yerel SSD diskine açar (~5-10 saniye sürer).* 

In [ ]:
import os

cache_zip = "/content/drive/MyDrive/BitirmeProjesi/autsl_10_caches.zip"
local_cache_root = "/content/BitirmeProjesi/cache"

if os.path.exists(cache_zip):
    os.makedirs(local_cache_root, exist_ok=True)
    print(f"📦 Önbellek paketi açılıyor: {cache_zip}")
    !unzip -q -o {cache_zip} -d {local_cache_root}
    
    print("\n--- Açılan Önbellek Dosya Sayıları ---")
    !echo "Temel Cache (train): $(ls /content/BitirmeProjesi/cache/autsl_10_features/train/*.npz 2>/dev/null | wc -l) (Beklenen: 1245)"
    !echo "Temel Cache (val):   $(ls /content/BitirmeProjesi/cache/autsl_10_features/val/*.npz 2>/dev/null | wc -l) (Beklenen: 199)"
    !echo "Aday Cache (train):  $(ls /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict/train/*.npz 2>/dev/null | wc -l) (Beklenen: 1245)"
    !echo "Aday Cache (val):    $(ls /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict/val/*.npz 2>/dev/null | wc -l) (Beklenen: 199)"
    print("✅ Önbellekler başarıyla yerel SSD'ye açıldı!")
else:
    print(f"❌ HATA: {cache_zip} bulunamadı! Lütfen autsl_10_caches.zip dosyasının Drive'daki BitirmeProjesi klasöründe olduğundan emin olun.")

### 4. 🔍 Preflight Doğrulama (Bütünlük ve Girdi Kontrolü)
Eğitime başlamadan önce tensor boyutlarını, etiket eşlemelerini ve her iki önbelleğin eksiksiz olduğunu doğrular.

In [ ]:
%cd /content/BitirmeProjesi

metadata_dir = "/content/BitirmeProjesi/data/autsl"
if not os.path.exists(os.path.join(metadata_dir, "val_labels.csv")):
    metadata_dir = "/content/drive/MyDrive/BitirmeProjesi/outputs/colab_training_metadata"

os.environ["AUTSL_DATA_DIR"] = metadata_dir

# Checksum dosyası varsa kontrolü dahil et
checksum_path = "/content/drive/MyDrive/BitirmeProjesi/outputs/colab_cache_checksums.json"
checksum_flag = f"--checksum_manifest {checksum_path}" if os.path.exists(checksum_path) else ""

!python scripts/check_colab_inputs.py \
    --metadata_dir {metadata_dir} \
    --base_cache /content/BitirmeProjesi/cache/autsl_10_features \
    --candidate_cache /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    {checksum_flag} \
    --output outputs/colab_preflight_colab.json

print("\n✅ Preflight denetimi tamamlandı! Eğitim aşamasına geçilebilir.")

### 5. 🚀 Deney 1: Temel 16 Kare Model Eğitimi (`autsl_10_base16_screen`)
- 20 Epoch, Seed 42, AdamW, Cosine Annealing, Dondurulmuş MobileNetV3
- T4 GPU ile yaklaşık **5-6 dakika** sürer.

In [ ]:
%cd /content/BitirmeProjesi

!python scripts/train.py \
    --config configs/experiment/autsl_10_base16_screen.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features

### 6. 🚀 Deney 2: Merkez Onarımlı Aday Model Eğitimi (`autsl_10_centerrepair_screen`)
- Bire bir aynı mimari, aynı tohum (`seed=42`) ve 20 Epoch
- T4 GPU ile yaklaşık **5-6 dakika** sürer.

In [ ]:
%cd /content/BitirmeProjesi

!python scripts/train.py \
    --config configs/experiment/autsl_10_centerrepair_screen.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict

### 7. 📊 Resmi Val Seti Üzerinde Karşılaştırmalı Değerlendirme
Eğitilen iki modelin `best_model.pt` ağırlıkları 199 val videosu üzerinde değerlendirilir.

In [ ]:
%cd /content/BitirmeProjesi

# 1. Temel Model Değerlendirmesi
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_base16_screen.yaml \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_base16_screen/checkpoints/best_model.pt \
    --split val \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features \
    --output outputs/base16_val_v2.json

# 2. Merkez Onarımlı Model Değerlendirmesi
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_centerrepair_screen.yaml \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_centerrepair_screen/checkpoints/best_model.pt \
    --split val \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    --output outputs/centerrepair_val_v2.json

### 8. 📈 Karşılaştırmalı Sonuç Tablosu

In [ ]:
import os, json
import pandas as pd

base_path = 'outputs/base16_val.json' if os.path.exists('outputs/base16_val.json') else 'outputs/base16_val_v2.json'
cand_path = 'outputs/centerrepair_val.json' if os.path.exists('outputs/centerrepair_val.json') else 'outputs/centerrepair_val_v2.json'

if os.path.exists(base_path) and os.path.exists(cand_path):
    b_res = json.load(open(base_path))
    c_res = json.load(open(cand_path))
    
    df = pd.DataFrame([
        {'Model / Varyant': '1. Temel 16 Kare (base16)', 'Val Top-1 Doğruluk': f"%{b_res['accuracy']*100:.2f}", 'Val Macro F1': f"{b_res['f1_macro']:.4f}", 'Top-5': f"%{b_res.get('top5_accuracy', 0)*100:.2f}"},
        {'Model / Varyant': '2. Merkez Onarımlı (centerrepair_strict)', 'Val Top-1 Doğruluk': f"%{c_res['accuracy']*100:.2f}", 'Val Macro F1': f"{c_res['f1_macro']:.4f}", 'Top-5': f"%{c_res.get('top5_accuracy', 0)*100:.2f}"}
    ])
    display(df)
else:
    print('Sonuç dosyaları henüz oluşmadı. Lütfen 7. adımı tamamlayın.')


### 9. 🔍 Seed 42: Video Bazlı Tahminlerin Çıkarılması (Hata Analizi İçin)
*Mevcut en iyi modellerin her video için doğru/yanlış tahminlerini JSON olarak kaydeder (~1-2 dakika sürer).*

In [ ]:
%cd /content/BitirmeProjesi
import os
metadata_dir = '/content/BitirmeProjesi/data/autsl'
if not os.path.exists(os.path.join(metadata_dir, 'val_labels.csv')):
    metadata_dir = '/content/drive/MyDrive/BitirmeProjesi/outputs/colab_training_metadata'
os.environ['AUTSL_DATA_DIR'] = metadata_dir

# 1. Temel Model (Seed 42) Video Tahminleri
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_base16_screen.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_base16_screen/checkpoints/best_model.pt \
    --split val \
    --output outputs/base16_seed42_val_verified.json \
    --predictions_output outputs/base16_seed42_predictions.json

# 2. Merkez Onarımlı Model (Seed 42) Video Tahminleri
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_centerrepair_screen.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_centerrepair_screen/checkpoints/best_model.pt \
    --split val \
    --output outputs/centerrepair_seed42_val_verified.json \
    --predictions_output outputs/centerrepair_seed42_predictions.json

print("\n✅ Seed 42 video bazlı tahminler başarıyla kaydedildi!")


### 10. 🎲 Seed 7 Doğrulama Eğitimi (Temel ve Merkez Onarımlı)
*Başarının tesadüf olmadığını kanıtlamak için model farklı bir rastgele tohumla (`seed=7`) 20 epoch eğitilir.*

In [ ]:
%cd /content/BitirmeProjesi

# Preflight kontrolü
!python scripts/check_colab_inputs.py \
    --base_config configs/experiment/autsl_10_base16_seed7.yaml \
    --candidate_config configs/experiment/autsl_10_centerrepair_seed7.yaml \
    --base_cache /content/BitirmeProjesi/cache/autsl_10_features \
    --candidate_cache /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    --metadata_dir "$AUTSL_DATA_DIR" \
    --output outputs/seed7_preflight_colab.json

# 1. Temel Model (Seed 7) Eğitimi
!python scripts/train.py \
    --config configs/experiment/autsl_10_base16_seed7.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features

# 2. Merkez Onarımlı Model (Seed 7) Eğitimi
!python scripts/train.py \
    --config configs/experiment/autsl_10_centerrepair_seed7.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict


### 11. 📊 Seed 7 Resmi Val Seti Değerlendirmesi ve Tahminleri

In [ ]:
%cd /content/BitirmeProjesi

# 1. Temel Model (Seed 7) Değerlendirmesi
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_base16_seed7.yaml \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_base16_seed7/checkpoints/best_model.pt \
    --split val \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features \
    --output outputs/base16_seed7_val.json \
    --predictions_output outputs/base16_seed7_predictions.json

# 2. Merkez Onarımlı Model (Seed 7) Değerlendirmesi
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_centerrepair_seed7.yaml \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_centerrepair_seed7/checkpoints/best_model.pt \
    --split val \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    --output outputs/centerrepair_seed7_val.json \
    --predictions_output outputs/centerrepair_seed7_predictions.json


### 12. 📈 Seed 7 Karşılaştırmalı Sonuç Tablosu

In [ ]:
import os, json
import pandas as pd

b_path = 'outputs/base16_seed7_val.json'
c_path = 'outputs/centerrepair_seed7_val.json'

if os.path.exists(b_path) and os.path.exists(c_path):
    b7 = json.load(open(b_path))
    c7 = json.load(open(c_path))
    
    df_seed7 = pd.DataFrame([
        {'Model (Seed 7)': '1. Temel 16 Kare (base16)', 'Val Top-1 Doğruluk': f"%{b7['accuracy']*100:.2f}", 'Val Macro F1': f"{b7['f1_macro']:.4f}", 'Top-5': f"%{b7.get('top5_accuracy', 0)*100:.2f}"},
        {'Model (Seed 7)': '2. Merkez Onarımlı (centerrepair_strict)', 'Val Top-1 Doğruluk': f"%{c7['accuracy']*100:.2f}", 'Val Macro F1': f"{c7['f1_macro']:.4f}", 'Top-5': f"%{c7.get('top5_accuracy', 0)*100:.2f}"}
    ])
    display(df_seed7)
else:
    print('Seed 7 sonuç dosyaları henüz oluşmadı. Lütfen 11. adımı tamamlayın.')


### 13. 🔬 Teşhis Deneyi: Pose-Only (Sadece İskelet) Seed 7 Eğitimi
*RGB el görüntülerini kapatıp modeli SADECE 225 iskelet noktasıyla eğitir (~4-5 dk sürer).*
*Amaç: İşaretçi 11 hatalarının el görüntüsünden mi yoksa iskeletten mi kaynaklandığını kesin olarak teşhis etmektir.*

In [ ]:
%cd /content/BitirmeProjesi
import os
os.environ['AUTSL_DATA_DIR'] = '/content/drive/MyDrive/BitirmeProjesi/outputs/colab_training_metadata'

# 1. Pose-Only Eğitimi (Seed 7) - 20 Epoch
!python scripts/train.py \
    --config configs/experiment/autsl_10_poseonly_seed7.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict

# 2. Pose-Only Değerlendirmesi ve Tahminleri
!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_poseonly_seed7.yaml \
    --cache_dir /content/BitirmeProjesi/cache/autsl_10_features_centerrepair_strict \
    --checkpoint /content/drive/MyDrive/BitirmeProjesi/outputs/autsl_10_poseonly_seed7/checkpoints/best_model.pt \
    --split val \
    --output outputs/poseonly_seed7_val.json \
    --predictions_output outputs/poseonly_seed7_predictions.json

print('\n✅ Pose-Only modeli eğitildi ve değerlendirildi!')


### 14. 📊 Teşhis Analizi: Hibrit vs Pose-Only Karşılaştırması
*İşaretçi 11'in hatalarının kaçının iskelet modeliyle düzeldiğini raporlar.*

In [ ]:
%cd /content/BitirmeProjesi

# Eşleştirilmiş ikili analiz
!python scripts/analyze_val_pairs.py \
    --base outputs/centerrepair_seed7_predictions.json \
    --candidate outputs/poseonly_seed7_predictions.json \
    --official_labels "$AUTSL_DATA_DIR/val_labels.csv" \
    --output outputs/hybrid_vs_poseonly_seed7_pairs.json

import os, json
import pandas as pd

p_path = 'outputs/poseonly_seed7_val.json'
c_path = 'outputs/centerrepair_seed7_val.json'

if os.path.exists(p_path) and os.path.exists(c_path):
    p_res = json.load(open(p_path))
    c_res = json.load(open(c_path))
    
    df_cmp = pd.DataFrame([
        {'Model Modu': '1. Hibrit Model (RGB El Kırpımı + İskelet)', 'Val Doğruluk': f"%{c_res['accuracy']*100:.2f}", 'Macro F1': f"{c_res['f1_macro']:.4f}", 'Top-5': f"%{c_res.get('top5_accuracy', 0)*100:.2f}"},
        {'Model Modu': '2. Pose-Only (Sadece İskelet Koordinatları)', 'Val Doğruluk': f"%{p_res['accuracy']*100:.2f}", 'Macro F1': f"{p_res['f1_macro']:.4f}", 'Top-5': f"%{p_res.get('top5_accuracy', 0)*100:.2f}"}
    ])
    display(df_cmp)
